# Debug Notebook: Fill Missing Pivot Rows (Quick Run)

This notebook fills missing rows in the Mode/Target/Draft time-breakdown pivot table using a small quick-run setup.

What it does:
- Reads existing deterministic quant summary rows from debug logs.
- Runs only missing profile combinations (or all if forced).
- Computes Drafting/Verify/Other time and shares in the same style as your table.
- Exports a final pivot CSV for direct copy into your spreadsheet.

In [3]:
import gc
import os
import sys
from pathlib import Path

import pandas as pd
try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "torch is required for this notebook. Select a kernel/environment with PyTorch installed."
    ) from exc

# Resolve project root robustly even when current working directory changes.
# Prefer searching from notebook location, then fallback to cwd ancestry.
search_roots = []
if "__file__" in globals():
    search_roots.append(Path(__file__).resolve().parent)
search_roots.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_roots:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / "src").exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError(
        "Could not find project root containing src/. Open this notebook under the speculative-decoding-main workspace."
    )

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Offline-first behavior.
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Force GPU-only execution.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for this notebook, but no GPU is available in the current kernel."
    )
if torch.cuda.device_count() < 1:
    raise RuntimeError("CUDA is available but no visible GPU device was found.")

from config import TARGET_MODEL_ID, DRAFT_MODELS, REGIMES
from baseline import run_baseline_sample
from speculative import load_model_on_device, speculative_decode_sample

debug_dir = project_root / "debug logs"
debug_dir.mkdir(parents=True, exist_ok=True)
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print("GPU-only mode enabled")
print(f"CUDA device count: {torch.cuda.device_count()}")
print("Forced device: cuda:0")

Project root: C:\Working\speculative-decoding-main
GPU-only mode enabled
CUDA device count: 2
Forced device: cuda:0


In [4]:
# Load existing deterministic summary rows from previous debug logs (if present).
summary_path = debug_dir / "quick_fast_quant_profile_summary_extended_10samples_0.5B_quick_k4_tok12.csv"
existing_rows = {}

if summary_path.exists():
    sdf = pd.read_csv(summary_path)
    for _, r in sdf.iterrows():
        mode = "Deterministic"
        target = str(r["target_quant_req"]).upper()
        draft = str(r["draft_quant_req"]).upper()
        spec_latency = float(r["mean_spec_latency_s"])
        draft_s = float(r["mean_draft_elapsed_s"])
        verify_s = float(r["mean_verify_elapsed_s"])
        other_s = max(spec_latency - draft_s - verify_s, 0.0)
        existing_rows[(mode, target, draft)] = {
            "Drafting (s)": draft_s,
            "Verify (s)": verify_s,
            "Other (s)": other_s,
            "Drafting %": float(r["spec_share_draft"]) * 100.0,
            "Verify %": float(r["spec_share_verify"]) * 100.0,
            "Other %": float(r["spec_share_other"]) * 100.0,
            "Source": "existing_debug_logs",
        }

print(f"Loaded existing rows: {len(existing_rows)}")
for k in sorted(existing_rows):
    print(" - ", k)

Loaded existing rows: 0


In [5]:
# Pivot template + quick-run controls.
TEMPLATE_ROWS = [
    # Deterministic: full 2x2 target/draft quant matrix
    ("Deterministic", "FP16", "FP16"),
    ("Deterministic", "FP16", "INT8"),
    ("Deterministic", "INT8", "FP16"),
    ("Deterministic", "INT8", "INT8"),

    # Stochastic: full 2x2 target/draft quant matrix
    ("Stochastic", "FP16", "FP16"),
    ("Stochastic", "FP16", "INT8"),
    ("Stochastic", "INT8", "FP16"),
    ("Stochastic", "INT8", "INT8"),
]

# Quick prompt set for fast debug runs (small, generic, and stable).
QUICK_PROMPTS = [
    "Why are leaves green? Answer in one sentence.",
    "Give three practical study tips in one short paragraph.",
    "Explain cloud computing for beginners in one sentence.",
    "Name three benefits of regular exercise briefly.",
    "What is an API? One-sentence answer.",
    "Difference between supervised and unsupervised learning in one sentence.",
    "Define overfitting in machine learning in one sentence.",
    "List two ways to reduce home energy consumption.",
    "What is version control in software development? One sentence.",
    "Provide one concise writing tip.",
]

# Clean rerun defaults: run every combination from scratch on a small sample set.
FORCE_RERUN = True       # True = run all combinations regardless of existing logs
N_SAMPLES = 5            # small sample size for quick clean rerun
K = 4
MAX_NEW_TOKENS = 12
DEVICE = "cuda:0"        # force GPU; no CPU fallback

# Hard guard to ensure we never run this notebook on CPU.
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU-only mode is enabled, but CUDA is not available. Use a GPU kernel/environment."
    )

# When FORCE_RERUN=True, final table should be composed only from newly generated rows.
USE_EXISTING_ROWS_FOR_FINAL = not FORCE_RERUN

prompt_subset = QUICK_PROMPTS[:N_SAMPLES]

if FORCE_RERUN:
    to_run = list(TEMPLATE_ROWS)
else:
    to_run = [row for row in TEMPLATE_ROWS if row not in existing_rows]

print(f"Rows in template: {len(TEMPLATE_ROWS)}")
print(f"Rows to run now: {len(to_run)}")
print(f"FORCE_RERUN={FORCE_RERUN}, N_SAMPLES={N_SAMPLES}, DEVICE={DEVICE}")
for r in to_run:
    print(" - ", r)

Rows in template: 8
Rows to run now: 8
FORCE_RERUN=True, N_SAMPLES=5, DEVICE=cuda:0
 -  ('Deterministic', 'FP16', 'FP16')
 -  ('Deterministic', 'FP16', 'INT8')
 -  ('Deterministic', 'INT8', 'FP16')
 -  ('Deterministic', 'INT8', 'INT8')
 -  ('Stochastic', 'FP16', 'FP16')
 -  ('Stochastic', 'FP16', 'INT8')
 -  ('Stochastic', 'INT8', 'FP16')
 -  ('Stochastic', 'INT8', 'INT8')


In [6]:
# Quick runner for one profile (Mode, Target quant, Draft quant).
target_cache = {}
draft_cache = {}

def _get_or_load_target(target_quant: str, device: str):
    key = (TARGET_MODEL_ID, target_quant.lower(), device)
    if key not in target_cache:
        model, tok = load_model_on_device(TARGET_MODEL_ID, device=device, quant_mode=target_quant.lower())
        target_cache[key] = (model, tok)
    return target_cache[key]

def _get_or_load_draft(draft_quant: str, device: str):
    model_id = DRAFT_MODELS["0.5B"]
    key = (model_id, draft_quant.lower(), device)
    if key not in draft_cache:
        model, tok = load_model_on_device(model_id, device=device, quant_mode=draft_quant.lower())
        draft_cache[key] = (model, tok)
    return draft_cache[key]

def run_profile(mode: str, target_quant: str, draft_quant: str, prompts: list[str]):
    regime_key = mode.lower()
    regime = REGIMES[regime_key]

    target_model, target_tok = _get_or_load_target(target_quant, DEVICE)
    draft_model, _ = _get_or_load_draft(draft_quant, DEVICE)

    per_sample = []
    for idx, prompt in enumerate(prompts, start=1):
        base = run_baseline_sample(
            target_model,
            target_tok,
            prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=regime.temperature,
            top_p=regime.top_p,
        )

        spec = speculative_decode_sample(
            target_model,
            draft_model,
            target_tok,
            prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            k=K,
            temperature=regime.temperature,
            top_p=regime.top_p,
            return_timing_breakdown=True,
        )

        draft_s = float(spec.get("draft_elapsed_s", 0.0))
        verify_s = float(spec.get("verify_elapsed_s", 0.0))
        spec_s = float(spec.get("latency_s", 0.0))
        other_s = max(spec_s - draft_s - verify_s, 0.0)

        per_sample.append({
            "sample_idx": idx,
            "baseline_latency_s": float(base.get("latency_s", 0.0)),
            "spec_latency_s": spec_s,
            "draft_elapsed_s": draft_s,
            "verify_elapsed_s": verify_s,
            "other_elapsed_s": other_s,
            "draft_share": (draft_s / spec_s) if spec_s > 0 else 0.0,
            "verify_share": (verify_s / spec_s) if spec_s > 0 else 0.0,
            "other_share": (other_s / spec_s) if spec_s > 0 else 0.0,
        })

    df = pd.DataFrame(per_sample)
    summary = {
        "Drafting (s)": float(df["draft_elapsed_s"].mean()),
        "Verify (s)": float(df["verify_elapsed_s"].mean()),
        "Other (s)": float(df["other_elapsed_s"].mean()),
        "Drafting %": float(df["draft_share"].mean() * 100.0),
        "Verify %": float(df["verify_share"].mean() * 100.0),
        "Other %": float(df["other_share"].mean() * 100.0),
        "Source": f"quick_run_n{len(prompts)}",
    }
    return summary, df

In [7]:
# Execute missing profiles and save per-sample debug CSVs.
new_rows = {}

for mode, target, draft in to_run:
    print(f"Running: mode={mode}, target={target}, draft={draft}, samples={len(prompt_subset)}")
    summary, df_samples = run_profile(mode, target, draft, prompt_subset)
    new_rows[(mode, target, draft)] = summary

    out_name = f"quick_missing_{mode.lower()}_{target.lower()}_target_{draft.lower()}_draft_n{len(prompt_subset)}.csv"
    out_path = debug_dir / out_name
    df_samples.to_csv(out_path, index=False)
    print(f"  saved sample debug -> {out_path.name}")

# Optional memory cleanup if you want to rerun with different settings.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Newly generated rows: {len(new_rows)}")

Running: mode=Deterministic, target=FP16, draft=FP16, samples=5
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

  saved sample debug -> quick_missing_stochastic_fp16_target_int8_draft_n5.csv
Running: mode=Stochastic, target=INT8, draft=FP16, samples=5


c:\ProgramData\anaconda3\envs\py12-dl\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  saved sample debug -> quick_missing_stochastic_int8_target_fp16_draft_n5.csv
Running: mode=Stochastic, target=INT8, draft=INT8, samples=5


c:\ProgramData\anaconda3\envs\py12-dl\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  saved sample debug -> quick_missing_stochastic_int8_target_int8_draft_n5.csv
Newly generated rows: 8


In [8]:
# Build final pivot table in screenshot-style columns and save CSV.
if USE_EXISTING_ROWS_FOR_FINAL:
    all_rows = {}
    all_rows.update(existing_rows)
    all_rows.update(new_rows)
else:
    # Clean mode: only use values generated in this run.
    all_rows = dict(new_rows)

final_records = []
for mode, target, draft in TEMPLATE_ROWS:
    key = (mode, target, draft)
    rec = {"Mode": mode, "Target": target, "Draft": draft}
    vals = all_rows.get(key)
    if vals is None:
        rec.update({
            "Drafting (s)": None,
            "Verify (s)": None,
            "Other (s)": None,
            "Drafting %": None,
            "Verify %": None,
            "Other %": None,
            "Source": "missing_after_clean_run" if FORCE_RERUN else "missing",
        })
    else:
        rec.update(vals)
    final_records.append(rec)

df_final = pd.DataFrame(final_records)

# Rounded display copy
df_view = df_final.copy()
for c in ["Drafting (s)", "Verify (s)", "Other (s)"]:
    df_view[c] = df_view[c].round(4)
for c in ["Drafting %", "Verify %", "Other %"]:
    df_view[c] = df_view[c].round(2)

display(df_view)

out_csv = results_dir / "pivot_time_breakdown_debug_quickrun.csv"
df_final.to_csv(out_csv, index=False)
print(f"Saved final pivot -> {out_csv}")

,Mode,Target,Draft,Drafting (s),Verify (s),Other (s),Drafting %,Verify %,Other %,Source
0,Deterministic,FP16,FP16,0.2291,0.1468,0.0163,58.79,36.73,4.49,quick_run_n5
1,Deterministic,FP16,INT8,1.4589,0.1457,0.0120,90.36,8.91,0.73,quick_run_n5
2,Deterministic,INT8,FP16,0.2286,1.0835,0.0140,17.71,81.25,1.04,quick_run_n5
3,Deterministic,INT8,INT8,1.5853,1.2009,0.0403,56.93,41.82,1.25,quick_run_n5
4,Stochastic,FP16,FP16,0.2371,0.1268,0.0144,62.90,33.32,3.78,quick_run_n5
5,Stochastic,FP16,INT8,1.4445,0.1531,0.0183,89.56,9.33,1.11,quick_run_n5
6,Stochastic,INT8,FP16,0.2479,1.0971,0.0234,18.66,79.65,1.70,quick_run_n5
7,Stochastic,INT8,INT8,1.5475,1.2027,0.0424,56.74,41.92,1.34,quick_run_n5


Saved final pivot -> C:\Working\speculative-decoding-main\results\pivot_time_breakdown_debug_quickrun.csv


## Notes

- This notebook is designed for quick debugging, not final benchmark reporting.
- Increase `N_SAMPLES` to 10 to get closer to the previous quick-run setup.
- Set `FORCE_RERUN=True` if you want to regenerate all rows from scratch.